# Health Score Calibration Notebook

This notebook calibrates Neurvinch health score weighting and threshold values using benchmark traces and scenario-based targets.

Goals:
- Tune contradiction/void penalty weights
- Tune contradiction probability threshold for F1
- Propose a retrieval void threshold based on query score distribution

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
OUT = ROOT / 'outputs'
EVAL = OUT / 'eval'

print('Root:', ROOT)
print('Outputs exists:', OUT.exists())
print('Eval exists:', EVAL.exists())

In [ ]:
report_path = OUT / 'report.json'
bench_path = EVAL / 'contradiction_benchmark.json'

report = json.loads(report_path.read_text(encoding='utf-8')) if report_path.exists() else {}
bench = json.loads(bench_path.read_text(encoding='utf-8')) if bench_path.exists() else {}

report

## 1) Weight Calibration via Scenario Targets

We fit weights for this score form:

score = 1 - (w_c * contradiction_rate + w_v * void_penalty)

where contradiction_rate = contradictions / chunks and void_penalty = min(1, void_clusters / 10).

In [ ]:
scenarios = pd.DataFrame([
    {'name': 'ideal', 'chunks': 100, 'contradictions': 0, 'void_clusters': 0, 'target': 1.00},
    {'name': 'good', 'chunks': 100, 'contradictions': 1, 'void_clusters': 0, 'target': 0.96},
    {'name': 'current', 'chunks': report.get('total_chunks', 91), 'contradictions': report.get('contradictions', 2), 'void_clusters': report.get('void_clusters', 0), 'target': report.get('knowledge_health_score', 0.9857)},
    {'name': 'degrading', 'chunks': 100, 'contradictions': 8, 'void_clusters': 2, 'target': 0.72},
    {'name': 'risky', 'chunks': 100, 'contradictions': 14, 'void_clusters': 5, 'target': 0.42},
])

def score_fn(chunks, contradictions, void_clusters, w_c):
    w_v = 1.0 - w_c
    contradiction_rate = contradictions / max(chunks, 1)
    void_penalty = min(1.0, void_clusters / 10.0)
    score = 1.0 - (w_c * contradiction_rate + w_v * void_penalty)
    return max(0.0, min(1.0, score))

grid = np.linspace(0.5, 0.95, 91)
rows = []
for w_c in grid:
    preds = scenarios.apply(lambda r: score_fn(r['chunks'], r['contradictions'], r['void_clusters'], w_c), axis=1)
    mse = float(np.mean((preds - scenarios['target']) ** 2))
    rows.append({'w_contradiction': float(w_c), 'w_void': float(1 - w_c), 'mse': mse})

weight_results = pd.DataFrame(rows).sort_values('mse').reset_index(drop=True)
weight_results.head(10)

## 2) Contradiction Threshold Calibration

We search thresholds from 0.5 to 0.99 using benchmark pair probabilities and optimize F1.

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

def threshold_sweep(backend_key):
    details = bench.get('details', {}).get(backend_key, {})
    pairs = details.get('pairs', [])
    if not pairs:
        return pd.DataFrame()

    y_true = np.array([int(p['label']) for p in pairs])
    probs = np.array([float(p['contradiction_probability']) for p in pairs])

    results = []
    for t in np.linspace(0.5, 0.99, 50):
        y_pred = (probs >= t).astype(int)
        results.append({
            'threshold': float(t),
            'precision': float(precision_score(y_true, y_pred, zero_division=0)),
            'recall': float(recall_score(y_true, y_pred, zero_division=0)),
            'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        })

    df = pd.DataFrame(results).sort_values('f1', ascending=False).reset_index(drop=True)
    return df

heur_df = threshold_sweep('heuristic')
xenc_df = threshold_sweep('cross_encoder')

print('Best heuristic threshold:')
display(heur_df.head(5))
print('Best cross-encoder threshold:')
display(xenc_df.head(5))

## 3) Void Threshold Recommendation

Estimate a retrieval void cutoff by looking at query best-score distribution.

Rule of thumb: pick around P20-P30 percentile if you want sensitive void detection, or P10 for stricter voids.

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'src'))

from neurvinch.config import settings
from neurvinch.indexing import StructuralIndexer
from neurvinch.retrieval import HybridRetriever

queries = pd.read_csv(ROOT / 'data' / 'queries' / 'query_logs.csv')['query'].dropna().tolist()
_, chunks = StructuralIndexer(settings.kb_path).index()
retriever = HybridRetriever(
    embedding_model_name=settings.embedding_model,
    embedding_backend=settings.embedding_backend,
    top_k_candidates=settings.top_k_candidates,
    top_k_final=settings.top_k_final,
)
retriever.fit(chunks)
scores = np.array([retriever.best_score(q) for q in queries], dtype=float)

void_candidates = {
    'p10': float(np.percentile(scores, 10)),
    'p20': float(np.percentile(scores, 20)),
    'p30': float(np.percentile(scores, 30)),
    'mean': float(np.mean(scores)),
}
void_candidates

## 4) Recommended Settings

Use the best rows above to set:
- `NEURVINCH_CONTRADICTION_THRESHOLD`
- `NEURVINCH_VOID_RETRIEVAL_THRESHOLD`
- health score weights in `knowledge_health_score` function